# Lab 5 – Phân phối cosine similarity theo điều kiện chụp

**Mục tiêu:** đo độ giống nhau (cosine similarity) giữa ảnh `reference` và các ảnh tự chụp ở nhiều điều kiện
(`normal`, `left`, `right`, `low_light`, `bright`, `far`, `near`, `expression`), rồi so với ngưỡng **0.7** của đề.

**Pipeline:** ảnh → MTCNN (phát hiện + crop mặt) → FaceNet/InceptionResnetV1 (embedding 512 chiều) → cosine similarity với embedding của ảnh reference.

**Cấu trúc thư mục cần có:**
```
lab_5/
├── data/
│   ├── reference/          # 1 ảnh tham chiếu
│   └── collected/
│       ├── metadata.csv
│       └── <condition>/    # normal, left, right, ...
└── notebook/               # notebook này nằm ở đây
```

**Đầu ra:** thư mục `lab_5/results/` gồm `similarity_results.csv` và `distribution.png`.

## 1. Import và cấu hình

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

THRESHOLD = 0.7  # ngưỡng của đề: sim > 0.7 -> Matched

# Dùng đường dẫn tuyệt đối (resolve) để không phụ thuộc thư mục làm việc của kernel
DATA = Path("../data").resolve()
assert DATA.exists(), f"Không thấy thư mục data: {DATA}  (kiểm tra vị trí notebook)"

# Thư mục lưu kết quả; tự tạo nếu chưa có (tránh lỗi FileNotFoundError khi lưu file)
RESULTS = DATA.parent / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

print("DATA   :", DATA)
print("RESULTS:", RESULTS)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\Users\\LENOVO\\OneDrive\\Tài liệu\\học\\computer vision\\lab_5\\results'

## 2. Khởi tạo mô hình

- **MTCNN** (`keep_all=False`): mỗi ảnh chỉ lấy **1 khuôn mặt tốt nhất** (vì ảnh tự chụp chỉ có một người).
- **FaceNet** (`InceptionResnetV1`, trọng số `vggface2`): biến mặt 160×160 thành vector 512 chiều.
  `.eval()` để tắt dropout/batchnorm ở chế độ huấn luyện.

In [ ]:
mtcnn = MTCNN(keep_all=False, device=device)
facenet = InceptionResnetV1(pretrained="vggface2").eval().to(device)


def embed(path):
    """Ảnh -> embedding 512 chiều (numpy). Trả về None nếu MTCNN không thấy mặt."""
    img = Image.open(path).convert("RGB")
    face = mtcnn(img)                      # tensor (3,160,160) hoặc None
    if face is None:
        return None
    with torch.no_grad():                  # không cần gradient khi suy luận
        emb = facenet(face.unsqueeze(0).to(device))   # thêm chiều batch -> (1,3,160,160)
    return emb.cpu().numpy()[0]


def cosine(a, b):
    """Cosine similarity giữa 2 vector, trong khoảng [-1, 1]."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 3. Embedding của ảnh reference

In [ ]:
# Lấy ảnh đầu tiên trong thư mục reference (jpg/jpeg/png)
ref_files = sorted(p for p in (DATA / "reference").iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert ref_files, "Thư mục reference đang trống"

ref_path = ref_files[0]
print("Reference:", ref_path.name)

ref_emb = embed(ref_path)
assert ref_emb is not None, "MTCNN không tìm thấy mặt trong ảnh reference"

## 4. Đọc metadata

`metadata.csv` **không có dòng tiêu đề**, nên đặt tên cột thủ công.
> ⚠️ Hai cột cuối (`brightness`, `mtcnn_prob`) là **suy đoán** – hãy đối chiếu với `collect_dataset.py` và sửa tên nếu cần.
> Phần tính similarity bên dưới chỉ dùng `filename` và `condition`, nên đặt tên sai không ảnh hưởng kết quả.

In [ ]:
cols = ["filename", "condition", "timestamp", "box_w", "box_h", "brightness", "mtcnn_prob"]
meta = pd.read_csv(DATA / "collected" / "metadata.csv", header=None, names=cols)

# Ảnh nằm trong thư mục con trùng tên điều kiện: collected/<condition>/<filename>
meta["path"] = [DATA / "collected" / c / f for c, f in zip(meta.condition, meta.filename)]
meta["exists"] = meta.path.map(Path.exists)

print("Tổng số dòng metadata:", len(meta))
print("Không tìm thấy file ảnh:", (~meta.exists).sum())
assert meta.exists.any(), "Không tìm thấy ảnh nào – kiểm tra lại cấu trúc thư mục collected/"

meta.condition.value_counts()

## 5. Tính similarity cho toàn bộ ảnh

Với mỗi ảnh: embedding → cosine với `ref_emb`.
- Không phát hiện được mặt hoặc thiếu file → `sim = NaN` (ghi nhận là *không detect*, không tính vào phân phối).
- `matched = sim > 0.7`; ảnh NaN được coi là **không match**.

In [ ]:
sims = []
for i, row in enumerate(meta.itertuples(), 1):
    e = embed(row.path) if row.exists else None
    sims.append(np.nan if e is None else cosine(ref_emb, e))
    if i % 50 == 0:
        print(f"  đã xử lý {i}/{len(meta)} ảnh")

meta["sim"] = sims
meta["detected"] = meta.sim.notna()      # MTCNN có thấy mặt không
meta["matched"] = meta.sim > THRESHOLD   # NaN > 0.7 -> False

# Lưu kết quả (đường dẫn tuyệt đối, utf-8-sig để Excel đọc đúng tiếng Việt)
meta.assign(path=meta.path.astype(str)).to_csv(
    RESULTS / "similarity_results.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", RESULTS / "similarity_results.csv")

## 6. Thống kê theo điều kiện

| Cột | Ý nghĩa |
|---|---|
| `n` | số ảnh |
| `detect_rate` | tỉ lệ ảnh MTCNN phát hiện được mặt |
| `sim_mean / median / min` | thống kê similarity (chỉ trên ảnh có mặt) |
| `matched_rate` | tỉ lệ ảnh vượt ngưỡng 0.7 trên **tổng** số ảnh (≈ 1 − FRR theo điều kiện) |

Sắp xếp theo `sim_mean` tăng dần: điều kiện khó nhất nằm trên cùng.

In [ ]:
summary = meta.groupby("condition").agg(
    n=("filename", "count"),
    detect_rate=("detected", "mean"),
    sim_mean=("sim", "mean"),
    sim_median=("sim", "median"),
    sim_min=("sim", "min"),
    matched_rate=("matched", "mean"),
).round(3).sort_values("sim_mean")

summary

## 7. Biểu đồ phân phối

- **Trái:** boxplot theo điều kiện – thấy điều kiện nào kéo similarity xuống.
- **Phải:** histogram toàn bộ – thấy phần lớn ảnh nằm trên hay dưới vạch 0.7.

In [ ]:
order = summary.index.tolist()
data = [meta.loc[meta.condition == c, "sim"].dropna() for c in order]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# --- Boxplot theo điều kiện ---
ax[0].boxplot(data)
ax[0].set_xticks(range(1, len(order) + 1))
ax[0].set_xticklabels(order, rotation=45)
ax[0].axhline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
ax[0].set_ylabel("Cosine similarity")
ax[0].set_title("Similarity theo điều kiện chụp")
ax[0].legend()

# --- Histogram tổng thể ---
ax[1].hist(meta.sim.dropna(), bins=30, color="steelblue", edgecolor="white")
ax[1].axvline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
ax[1].set_xlabel("Cosine similarity")
ax[1].set_ylabel("Số ảnh")
ax[1].set_title("Phân phối similarity (tất cả ảnh)")
ax[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / "distribution.png", dpi=150)
plt.show()

## 8. Các ảnh tệ nhất

10 ảnh có similarity thấp nhất, cộng với các ảnh MTCNN không phát hiện được mặt – dùng để minh họa lỗi trong báo cáo.

In [ ]:
print("Ảnh không phát hiện được mặt:", (~meta.detected).sum())
meta.loc[meta.detected].nsmallest(10, "sim")[["filename", "condition", "sim"]]

## 9. Nhận xét (điền sau khi chạy)

- Điều kiện nào có similarity thấp nhất / hay rớt dưới 0.7?
- Điều kiện nào MTCNN khó phát hiện mặt nhất?
- Ngưỡng 0.7 có phù hợp với dữ liệu tự chụp không? (nếu `matched_rate` thấp ở nhiều điều kiện → ngưỡng khá chặt)
- Lưu ý: bộ dữ liệu này chỉ có **ảnh của chính mình**, nên chưa đo được lỗi nhận nhầm người lạ (FAR) – phần đó thuộc notebook evaluation.